# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
#Creating our first agent:

#Creation of agent is simple. name_of_agent = Agent (name = "", instructions = "") is the general syntax.
nutrition_agent = Agent (
    name = "Nutrition Assistant",
    #Instructions for the agent can be as simple. It is like giving an LLM a prompt
    instructions = """
    You are a helpful assistant giving out nutrition advice.
    You are giving concise answers.
    """
)

Let's execute the Agent:

In [5]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit. They’re:
    
    - Rich in potassium, vitamin B6, vitamin C, and dietary fiber
    - Moderate in calories and naturally sweet
    - Good for heart health, digestion, and quick energy
    
    Notes:
    - Riper bananas have more sugar; unripe have more resistant starch (lower GI).
    - If you have kidney issues or potassium restrictions, monitor intake.
    - Pair with protein or fat for steady energy.
    
    Overall: a nutritious, everyday option in moderation.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [6]:
#The runner's run_streamed function does the reply chatbot like i.e. it prints the reply line by line.
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    #using event.type = "raw_response_event" you can print only the output and no debug messages and additional information.
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy and convenient. Key points:

- Nutrients: good source of potassium, vitamin C, vitamin B6, and dietary fiber.
- Benefits: support heart health, digestion, and steady energy from natural sugars.
- Considerations: moderate in sugar; portion matters for low-carb or diabetes goals.
- Tips: 1 medium banana (~110–120 kcal) fits most diets; ripe bananas are sweeter, firmer green ones are starchier.
- Storage: keep at room temperature; refrigerating slows ripening (peel may darken).

Overall, a healthy, balanced part of most diets in moderate amounts.

_Good Job!_